# Ep 03 - Structured Output with Pydantic

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


In [2]:
load_dotenv()

True

In [3]:
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL","ollama:llama3.2")
GROQ_MODEL = os.getenv("GROQ_MODEL","Groq:meta-llama-prompt-guard-2-22m")

print("Ollama model: ", OLLAMA_MODEL)
print("Groq model: ", GROQ_MODEL)

Ollama model:  ollama:llama3.2
Groq model:  groq:qwen/qwen3.8-27b


## Output structure format using Pydantic

In [4]:
class Resume(BaseModel):
    name:str=Field(description="Full name of the candidate")
    email:str | None = Field(description="Get email if present, else none")
    years_experience:float = Field(description="Total years of professional experience")
    skills:list[str] = Field(description="List of technical skills mentioned")
    

In [5]:
Resume(name="raj",email="raj@gmail.com",years_experience=5.9,skills=["python","langchain","langgraph"])

Resume(name='raj', email='raj@gmail.com', years_experience=5.9, skills=['python', 'langchain', 'langgraph'])

## Load input data

In [6]:
SAMPLE_DIR = Path("samples")

TEXT = (SAMPLE_DIR / "resume_messy.txt").read_text(encoding="utf-8")

In [7]:
print(TEXT)

hey so my name is Aarav Sharma, been working as a backend dev for about 4 and a half
years now. you can reach me at aarav.sharma@example.com or just ping on linkedin.
mostly i do python, fastapi, postgres, some docker and aws. recently started learning
langgraph and building ai agents which is super fun. also know a bit of react but not my
main thing. looking for agentic ai engineer roles now.


## Parse resume using LLM

In [8]:
llm=init_chat_model(OLLAMA_MODEL, temperature=0) #


In [9]:
prompt = f"From given resume{TEXT} \n extract name, email, years of experience and skills in json format."
responce = llm.invoke(prompt)

In [10]:
responce

AIMessage(content='Here is the extracted information in JSON format:\n\n```\n{\n  "name": "Aarav Sharma",\n  "email": "aarav.sharma@example.com",\n  "years_of_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "React"\n  ]\n}\n```\n\nNote: I\'ve assumed that the 4.5 years of experience is a rough estimate, as the problem statement doesn\'t specify the exact start and end dates of your employment. If you have more precise information, feel free to let me know and I can update the JSON accordingly.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-24T12:51:14.1516405Z', 'done': True, 'done_reason': 'stop', 'total_duration': 23952111900, 'load_duration': 21730100, 'prompt_eval_count': 141, 'prompt_eval_duration': 182123000, 'eval_count': 146, 'eval_duration': 23671700000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0d377-f8d1-7690

In [11]:
responce.content

'Here is the extracted information in JSON format:\n\n```\n{\n  "name": "Aarav Sharma",\n  "email": "aarav.sharma@example.com",\n  "years_of_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "React"\n  ]\n}\n```\n\nNote: I\'ve assumed that the 4.5 years of experience is a rough estimate, as the problem statement doesn\'t specify the exact start and end dates of your employment. If you have more precise information, feel free to let me know and I can update the JSON accordingly.'

In [12]:
llm_structure = llm.with_structured_output(Resume,include_raw=True)

In [ ]:
# llm_structure.invoke(prompt)

## Messages: System vs User

System message = the agent's role/rules ("You are a percise data extractor.).user message = the actual request/content. Good system prompt = more reliable output. (We'll harden these as guardrails in Ep 35.)

In [13]:
system = "You are a precise data extractor. Extract only what is present; never invent data."

msg = [
    {"role": "system", "content": system},
    {"role": "user", "content": f"Extract structured data from this resume : \n {TEXT}"}
]

responce = llm_structure.invoke(msg)

In [14]:
responce

{'raw': AIMessage(content='{\n  "name": "Aarav Sharma",\n  "email": "aarav.sharma@example.com",\n  "years_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS"\n  ]\n}', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-24T13:09:27.079796Z', 'done': True, 'done_reason': 'stop', 'total_duration': 44585443900, 'load_duration': 24927132000, 'prompt_eval_count': 148, 'prompt_eval_duration': 5828671000, 'eval_count': 65, 'eval_duration': 13725784000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0d388-557b-72a2-98bd-689e51a6b2ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, 'output_tokens': 65, 'total_tokens': 213}),
 'parsed': Resume(name='Aarav Sharma', email='aarav.sharma@example.com', years_experience=4.5, skills=['Python', 'FastAPI', 'Postgres', 'Docker', 'AWS']),
 'parsing_error': None}

In [15]:
responce["parsed"]

Resume(name='Aarav Sharma', email='aarav.sharma@example.com', years_experience=4.5, skills=['Python', 'FastAPI', 'Postgres', 'Docker', 'AWS'])

## TOKEN/COUNT Awareness

In [ ]:
{'raw': AIMessage(content='{\n  "name": "Aarav Sharma",\n  "email": "aarav.sharma@example.com",\n  "years_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS"\n  ]\n}', additional_kwargs={}, 
response_metadata={'model': 'llama3.2', 'created_at': '2026-09-24T13:09:27.079796Z', 'done': True, 'done_reason': 'stop', 'total_duration': 44585443900, 
'load_duration': 24927132000, 'prompt_eval_count': 148, 'prompt_eval_duration': 5828671000, 'eval_count': 65, 'eval_duration': 13725784000, 'logprobs': None, 
'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0d388-557b-72a2-98bd-689e51a6b2ce-0', tool_calls=[], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 148, 'output_tokens': 65, 'total_tokens': 213}),
 'parsed': Resume(name='Aarav Sharma', email='aarav.sharma@example.com', years_experience=4.5, skills=['Python', 'FastAPI', 'Postgres', 'Docker', 'AWS']),
 'parsing_error': None}

In [18]:
usage = getattr(responce.get("raw"), "usage_metadata", None)
if usage:
    print(f"token - in: {usage.get("input_tokens")} out: {usage.get("output_token")}"
            f"total: {usage.get("total_tokens")}")


token - in: 148 out: Nonetotal: 213


## Never trust unvalidated output

In [20]:
if responce.get("parsing_error"):
    print("Model output failed validation; handle/retry instead of trusting it.")
else:
    resume = responce.get("parsed")
    print(resume.model_dump_json(indent=2))
    print(f"\nValidated! {len(resume.skills)} skills, {resume.years_experience} yrs experience.")

{
  "name": "Aarav Sharma",
  "email": "aarav.sharma@example.com",
  "years_experience": 4.5,
  "skills": [
    "Python",
    "FastAPI",
    "Postgres",
    "Docker",
    "AWS"
  ]
}

Validated! 5 skills, 4.5 yrs experience.
